In [12]:
pip install xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ------- -------------------------------- 12.6/69.5 MB 60.7 MB/s eta 0:00:01
   --------------- ------------------------ 27.5/69.5 MB 67.2 MB/s eta 0:00:01
   ------------------------ --------------- 42.2/69.5 MB 70.6 MB/s eta 0:00:01
   -------------------------------------- - 67.4/69.5 MB 82.6 MB/s eta 0:00:01
   ---------------------------------------- 69.5/69.5 MB 75.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

CC = pd.read_csv(r"C:\Users\Afdal Hussain\Desktop\Python Projects\Datasets\creditcard.csv")

CC.describe

<bound method NDFrame.describe of             Time         V1         V2        V3        V4        V5  \
0            0.0  -1.359807  -0.072781  2.536347  1.378155 -0.338321   
1            0.0   1.191857   0.266151  0.166480  0.448154  0.060018   
2            1.0  -1.358354  -1.340163  1.773209  0.379780 -0.503198   
3            1.0  -0.966272  -0.185226  1.792993 -0.863291 -0.010309   
4            2.0  -1.158233   0.877737  1.548718  0.403034 -0.407193   
...          ...        ...        ...       ...       ...       ...   
284802  172786.0 -11.881118  10.071785 -9.834783 -2.066656 -5.364473   
284803  172787.0  -0.732789  -0.055080  2.035030 -0.738589  0.868229   
284804  172788.0   1.919565  -0.301254 -3.249640 -0.557828  2.630515   
284805  172788.0  -0.240440   0.530483  0.702510  0.689799 -0.377961   
284806  172792.0  -0.533413  -0.189733  0.703337 -0.506271 -0.012546   

              V6        V7        V8        V9  ...       V21       V22  \
0       0.462388  0.239599

In [4]:
y = CC["Class"]
x = CC.drop(columns=["Class"])

In [31]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=500
)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [41]:
models = {
    "LR Baseline":
    LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    ),

    "RF Baseline":
    RandomForestClassifier(
        random_state=42
    ),

    "RF Tuned":
    RandomForestClassifier(
        n_estimators=350,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost Baseline":
    XGBClassifier(
        random_state=42
    ),

    "XGBoost Tuned":
    XGBClassifier(
        n_estimators=350,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=2,
        gamma=1,
        reg_alpha=0.1,
        reg_lambda=1,
        eval_metric='aucpr',
        n_jobs=-1,
        random_state=42
    )
}

In [42]:
for i, model in models.items():
    model.fit(x_train_scaled, y_train)
    y_pred = model.predict(x_test_scaled)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(i, precision, recall, f1)

LR Baseline 0.06456346294937637 0.9072164948453608 0.12054794520547946
RF Baseline 0.9375 0.7731958762886598 0.847457627118644
RF Tuned 0.9069767441860465 0.8041237113402062 0.8524590163934426
XGBoost Baseline 0.9493670886075949 0.7731958762886598 0.8522727272727273
XGBoost Tuned 0.9397590361445783 0.8041237113402062 0.8666666666666667
